# Bayesian Phase Coordinates Demo

This notebook demonstrates `fit_bayesian_phase_coordinates`, which estimates phase coordinates using a two-layer Bayesian model with MCMC sampling.

**Note:** MCMC sampling takes ~10-60 seconds for this modest example. Real data with more cycles and draws will take longer. Requires `pip install phase-coordinates[bayes]`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from phase_coordinates import fit_bayesian_phase_coordinates, reconstruct_phase_coordinates

In [ ]:
rng = np.random.default_rng(0)
fs = 100.0
n_per_cycle = 100
n_cycles = 4
n = n_per_cycle * n_cycles
t = np.arange(n) / fs
tilt = np.pi / 6
X = np.column_stack([
    np.cos(2 * np.pi * t),
    np.sin(2 * np.pi * t) * np.cos(tilt),
    np.sin(2 * np.pi * t) * np.sin(tilt),
])
X += rng.normal(0, 0.02, X.shape)
print(f"X shape: {X.shape}, ~{n_cycles} cycles at {fs} Hz")

In [ ]:
# Note: MCMC takes ~10-60s for this modest example.
samples, cycles, details = fit_bayesian_phase_coordinates(
    X, sampling_rate_hz=fs, draws=400, tune=400, chains=2, random_seed=0
)
print("samples shape:", samples.shape)
print("cycles shape:", cycles.shape)
print("algorithm:", details["algorithm"])
print("sigma_x_mean:", details["uncertainty"]["sigma_x_mean"])

In [ ]:
X_hat = reconstruct_phase_coordinates(samples, cycles)
fitted = ~np.isnan(X_hat[:, 0])
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
for i, label in enumerate(["x", "y", "z"]):
    axes[i].plot(X[:, i], lw=0.8, alpha=0.5, label="observed")
    axes[i].plot(X_hat[:, i], lw=1.5, label="posterior mean")
    axes[i].set_ylabel(label)
axes[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
print("Fitted cycles:")
print(cycles[["cycle", "n_samples", "radius_mean", "fit_ok"]])
print("\nDiagnostics:")
diag = details["diagnostics"]
print("  failures:", diag["failures"])
print("  sigma_x/R_X:", diag["sigma_x_over_RX"])
print("  phase_monotonic:", diag["phase_monotonic"])

**Current known limitations of fit_bayesian_phase_coordinates:**
- Endpoint boundary drift: the first and last cycle boundaries can drift 3-5 samples from the true cycle start/end, inflating residuals in the first and last cycles.
- Linear phase within cycle: the model assumes linear phase within each cycle, which inflates sigma_x when within-cycle speed is non-uniform.
- MCMC runtime: a 400-sample, 4-cycle run takes ~10-60s; real data with more cycles and draws will take longer.